## Decoder Layer

論文: Vaswani et al. (2017), Section 3.1 / Figure 1(右側がデコーダ)

<img src="decoder_v2.png" width="240">

デコーダ層は、出力系列の各トークンのベクトルを更新する層。エンコーダ層(2 サブ層)に **Cross-Attention** を 1 つ足した **3 サブ層**でできている。

### 入力の形

デコーダ層には **2 つの入力**が入る。どちらも形は `(batch, seq, d_model)`。

| 入力 | 形 | 中身 |
|---|---|---|
| `x` | `(batch, tgt_seq, d_model)` | デコーダ側の系列(生成中の出力)。`tgt_seq` は出力の長さ |
| `enc_output` | `(batch, src_seq, d_model)` | エンコーダの出力。`src_seq` は入力文の長さ |

さらに 2 つのマスクを受け取り、使う場所が違う。

| マスク | 使う場所 | 中身 |
|---|---|---|
| `tgt_mask` | ステップ1(Self-Attention) | パディング + 因果マスク(未来を隠す) |
| `src_mask` | ステップ2(Cross-Attention) | パディングのみ |

因果マスクは下のセルの `make_causal_mask` が作る(`torch.tril` で下三角=見てよい位置が 1)。

### このlayerがやること

デコーダの各トークンのベクトルを、**(1)すでに生成した出力トークン と (2)エンコーダが作った入力文 の情報を取り込んだベクトルに更新する**。入力と同じ形 `(batch, seq, d_model)` で返す。

更新は 3 ステップ。

**ステップ1: Masked Self-Attention(02 で実装済み)**
各トークンが、同じ出力系列の**自分より前のトークンだけ**を見て、関係の深い相手のベクトルを重みづけして取り込む。`tgt_mask`(因果マスク)で未来の位置を隠す。
→ デコーダは左から 1 トークンずつ生成するので、まだ無い未来は見られない。学習時もマスクで未来を隠し、答えの先読みを防ぐ。

**ステップ2: Cross-Attention(エンコーダ層には無い)**
各トークン(Query)が、**エンコーダ出力 `enc_output`(Key・Value)**を見て情報を取り込む。
→ 入力文の情報がデコーダに入る唯一の場所。出力中のトークンが入力文のどこを参照するかを決める。

**ステップ3: Feed Forward(03 で実装済み)**
ステップ1・2で情報を取り込んだ各トークンを、1 個ずつ独立に変換する(他のトークンは見ない)。

各ステップの出力は、05 と同じく **Add & Norm**(残差接続 + LayerNorm)で包む。

### 流れ

入力 `x`(形 `(batch, tgt_seq, d_model)`)と Encoder 出力 `enc_output` を、図の下のブロックから順に通す。

```
# サブ層1: Masked Self-Attention → Add & Norm
a = self_attn(x, x, x, tgt_mask)        # Q,K,V は同じ x。tgt_mask で未来を隠す
x = norm1(x + dropout1(a))

# サブ層2: Cross-Attention → Add & Norm
a = cross_attn(x, enc_output, enc_output, src_mask)   # Q=x, K=V=enc_output
x = norm2(x + dropout2(a))

# サブ層3: Feed Forward → Add & Norm
f = ffn(x)
x = norm3(x + dropout3(f))

return x
```

- 各サブ層の出力は、`x` に足す前に dropout をかける
- 入力も出力も形は `(batch, tgt_seq, d_model)` のまま。だから図の **N×** のように同じ層を何段も積める(論文では 6 段)

In [ ]:
import sys
sys.path.append("..")

import torch
import torch.nn as nn
from src.attention import MultiHeadAttention
from src.feed_forward import PositionwiseFeedForward


In [ ]:
# 因果マスクを作る関数。
def make_causal_mask(seq):
    # torch.tril(...) : 下三角だけ1、右上(未来側)を0にする → 見てよい位置=1, 見てはいけない位置=0
    mask = torch.tril(torch.ones(seq, seq)).unsqueeze(0).unsqueeze(0) #(1, 1, seq, seq) → (batch, num_heads, seq, seq) のattentionスコアにブロードキャストで合わせるため
    return mask

print(make_causal_mask(5))

tensor([[[[1., 0., 0., 0., 0.],
          [1., 1., 0., 0., 0.],
          [1., 1., 1., 0., 0.],
          [1., 1., 1., 1., 0.],
          [1., 1., 1., 1., 1.]]]])


In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        # 3つのサブレイヤー
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)

        # 3つのLayerNorm
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

        # 3つのDropout
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x, enc_output, src_mask=None, tgt_mask=None): 
        #src_mask：パディングだけ　　tgt_mask：パディング＋因果マスク
        # x : (batch, seq, d_model)←このseqはデコーダーの入力   enc_output : (batch, seq, d_model)←このseqはエンコーダーの出力の長さ
        
        # サブレイヤー1: Masked Self-Attention
        attn_output = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout1(attn_output))

        # サブレイヤー2: Cross-Attention
        # Query=x (Decoderの中間表現), Key=Value=enc_output
        attn_output = self.cross_attn(x, enc_output, enc_output, src_mask)
        x = self.norm2(x + self.dropout2(attn_output))

        # サブレイヤー3: Feed-Forward Network
        ffn_output = self.ffn(x)
        x = self.norm3(x + self.dropout3(ffn_output))
        return x

In [ ]:
class Decoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([    
            DecoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)   #層を作るループ
        ])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
        for layer in self.layers:                        #作った層にxを流す
            x = layer(x, enc_output, src_mask, tgt_mask)
        return self.norm(x)

In [ ]:
import torch

# Decoderのテスト
decoder = Decoder(d_model=512, num_heads=8, d_ff=2048, num_layers=6)
# Decoder入力とEncoder出力
tgt = torch.randn(2, 15, 512)   # 出力系列（長さ15）
enc_out = torch.randn(2, 20, 512)  # Encoder出力（長さ20）
# 因果マスク
tgt_mask = make_causal_mask(15)

output = decoder(tgt, enc_out, tgt_mask=tgt_mask)
print("Decoder入力形状:", tgt.shape)
print("Encoder出力形状:", enc_out.shape)
print("Decoder出力形状:", output.shape)


Decoder入力形状: torch.Size([2, 15, 512])
Encoder出力形状: torch.Size([2, 20, 512])
Decoder出力形状: torch.Size([2, 15, 512])
